In [1]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re

import pandas as pd

### Integrating numerical toxicity properties into MAOMAO

This notebook cross-references the usable HC50, LC50, LD50, and MHC measurements with the MAOMAO sequence pivot. It preserves every experimental measurement in a canonical long table and creates a separate sequence-level wide table for characterization.

It does **not** modify `maomao_sequence_pivot.csv` and does **not** silently append sequences that are absent from the master pivot. Unmatched sequences are retained in the long table and exported to an audit file. Units are not converted or combined.

Generated files in `processed_data/processed_data/`:

- `maomao_toxicity_measurements.csv`: one row per numerical toxicity measurement.
- `maomao_sequence_pivot_with_toxicity_properties.csv`: the master pivot plus four compact fields: `hc50`, `lc50`, `ld50`, and `mhc`. A missing measurement is encoded as `999`.
- `audit_toxicity_property_cross_reference.csv`: cross-reference counts by property.
- `audit_toxicity_property_unmatched_sequences.csv`: measurements whose sequences are absent from the master pivot.
- `metadata_toxicity_properties.json`: processing configuration and output summary.

- Configuration: The repository root is detected automatically when the notebook is executed from anywhere inside the MAOMAO repository. If the notebook is stored elsewhere, set the `MAOMAO_ROOT` environment variable or pass the repository path explicitly to `find_repo_root()`. No user-specific path is required.

In [2]:
def find_repo_root(start=None):
    configured_root = start or os.environ.get("MAOMAO_ROOT") or Path.cwd()
    start_path = Path(configured_root).expanduser().resolve()

    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "processed_data").is_dir():
            return candidate

    raise FileNotFoundError(
        f"Could not locate the MAOMAO repository root from: {start_path}. "
        "Place this notebook inside the cloned MAOMAO repository and run it there, "
        "set the MAOMAO_ROOT environment variable, or call find_repo_root('/path/to/maomao')."
    )


REPO_ROOT = find_repo_root()
MASTER_PATH = REPO_ROOT / "processed_data" / "processed_data" / "maomao_sequence_pivot.csv"
MEASUREMENT_ROOT = (
    REPO_ROOT
    / "processed_data"
    / "toxic_effect_regression"
    / "Hemolytik2.0_2026"
)
OUTPUT_ROOT = REPO_ROOT / "processed_data" / "processed_data"

INPUT_FILES = {
    "HC50": MEASUREMENT_ROOT / "processed_HC50_dataset.csv",
    "LC50": MEASUREMENT_ROOT / "processed_LC50_dataset.csv",
    "LD50": MEASUREMENT_ROOT / "processed_LD50_dataset.csv",
    "MHC": MEASUREMENT_ROOT / "processed_MHC_dataset.csv",
}

SOURCE_DATASET = "Hemolytik2.0_2026"
MIN_SEQUENCE_LENGTH = 5
MAX_SEQUENCE_LENGTH = 70
CANONICAL_RESIDUES = set("ACDEFGHIKLMNPQRSTVWY")
REQUIRED_MEASUREMENT_COLUMNS = {"sequence", "source", "label", "unit"}

print("Repository root:", REPO_ROOT)
print("Master pivot:", MASTER_PATH)
print("Measurement directory:", MEASUREMENT_ROOT)
print("Output directory:", OUTPUT_ROOT)

Repository root: /home/nicole/Descargas/maomao
Master pivot: /home/nicole/Descargas/maomao/processed_data/processed_data/maomao_sequence_pivot.csv
Measurement directory: /home/nicole/Descargas/maomao/processed_data/toxic_effect_regression/Hemolytik2.0_2026
Output directory: /home/nicole/Descargas/maomao/processed_data/processed_data


- Normalization and measurement parsing 

The same sequence normalization and SHA-256 convention used by the MAOMAO master pivot are applied here. Reported values such as `1.4 ± 0.2`, `> 200`, and `< 5` are split into numeric value, error, and relation fields while the original label is retained.

In [3]:
MEASUREMENT_PATTERN = re.compile(
    r"^\s*(?P<relation>[<>])?\s*"
    r"(?P<value>\d+(?:\.\d+)?)"
    r"(?:\s*±\s*(?P<error>\d+(?:\.\d+)?))?\s*$"
)


def normalize_sequence(value):
    if pd.isna(value):
        return pd.NA
    sequence = re.sub(r"\s+", "", str(value)).upper()
    return sequence or pd.NA


def generate_sequence_id(sequence):
    digest = hashlib.sha256(sequence.encode("utf-8")).hexdigest()
    return f"sha256_{digest}"


def parse_measurement_label(label):
    match = MEASUREMENT_PATTERN.fullmatch(str(label))
    if match is None:
        return pd.Series(
            {"relation": pd.NA, "value": pd.NA, "error": pd.NA},
            dtype="object",
        )

    return pd.Series(
        {
            "relation": match.group("relation") or "=",
            "value": float(match.group("value")),
            "error": (
                float(match.group("error"))
                if match.group("error") is not None
                else pd.NA
            ),
        }
    )

- Load and validate the master pivot

In [4]:
master = pd.read_csv(MASTER_PATH, dtype={"id": "string", "sequence": "string"})
required_master_columns = {"id", "sequence"}
missing_master_columns = required_master_columns.difference(master.columns)
assert not missing_master_columns, f"Missing master columns: {sorted(missing_master_columns)}"

master["sequence_normalized"] = master["sequence"].map(normalize_sequence)
assert master["id"].notna().all(), "The master contains missing identifiers."
assert master["sequence_normalized"].notna().all(), "The master contains missing sequences."
assert master["id"].is_unique, "The master identifier column is not unique."
assert master["sequence_normalized"].is_unique, "The master contains duplicate normalized sequences."

expected_master_ids = master["sequence_normalized"].map(generate_sequence_id)
assert master["id"].equals(expected_master_ids.astype(master["id"].dtype)), (
    "Master identifiers do not match SHA-256 hashes of normalized sequences."
)

master_original_columns = [column for column in master.columns if column != "sequence_normalized"]
print(f"Master rows: {len(master):,}")
display(master[master_original_columns].head())

Master rows: 71,857


,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,anti_mammalian_cells,neurotoxic,embryotoxic,ichthyotoxic
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340a...,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,0,999,999,999
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d...,AAAAAAAAAGETS,999,0,0,999,0,999,999,999
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803...,AAAAAAAAAK,999,999,0,999,999,999,999,999
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb6...,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,0,999,999,999
4,sha256_42c3b829be38387546e22839e4d70387e2af55c...,AAAAARRRIRKQAHAHSK,0,0,0,999,0,999,999,999


- Load and standardize the numerical measurements

No unit conversion is performed. Combining `µM` and `µg/mL` would require an explicitly documented molecular-weight and chemical-form convention.

In [5]:
measurement_frames = []

for measurement_type, path in INPUT_FILES.items():
    data = pd.read_csv(path, dtype="string")
    missing_columns = REQUIRED_MEASUREMENT_COLUMNS.difference(data.columns)
    assert not missing_columns, f"{path.name}: missing columns {sorted(missing_columns)}"

    data = data.loc[:, ["sequence", "source", "label", "unit"]].copy()
    data.insert(0, "measurement_type", measurement_type)
    data["source_dataset"] = SOURCE_DATASET
    data = data.rename(columns={"label": "raw_label"})
    data["sequence"] = data["sequence"].map(normalize_sequence)

    parsed = data["raw_label"].apply(parse_measurement_label)
    data = pd.concat([data, parsed], axis=1)
    measurement_frames.append(data)

measurements = pd.concat(measurement_frames, ignore_index=True)
measurements["value"] = pd.to_numeric(measurements["value"], errors="coerce")
measurements["error"] = pd.to_numeric(measurements["error"], errors="coerce")
measurements["sequence_length"] = measurements["sequence"].str.len().astype("Int64")
measurements["is_canonical"] = measurements["sequence"].map(
    lambda sequence: bool(sequence) and set(sequence).issubset(CANONICAL_RESIDUES)
)
measurements["within_maomao_length"] = measurements["sequence_length"].between(
    MIN_SEQUENCE_LENGTH, MAX_SEQUENCE_LENGTH
)

unparsed = measurements[measurements[["relation", "value"]].isna().any(axis=1)]
assert unparsed.empty, (
    "Some numerical labels could not be parsed. Review these rows before continuing:\n"
    + unparsed.to_string(index=False)
)
assert measurements["is_canonical"].all(), (
    "Non-canonical sequences remain in the usable numerical measurement files."
)

print(f"Measurement rows: {len(measurements):,}")
print(f"Unique measurement sequences: {measurements['sequence'].nunique():,}")
display(
    measurements.groupby(["measurement_type", "unit"], as_index=False)
    .agg(records=("sequence", "size"), unique_sequences=("sequence", "nunique"))
)

Measurement rows: 169
Unique measurement sequences: 166


,measurement_type,unit,records,unique_sequences
0,HC50,µM,39,38
1,HC50,µg/mL,5,5
2,LC50,µM,59,59
3,LD50,µM,21,21
4,MHC,µM,14,14
5,MHC,µg/mL,31,31


- Cross-reference against the master pivot

The identifier is deterministically generated from each normalized sequence. For matched records, the notebook verifies that it is exactly the identifier already stored in the master pivot.

In [6]:
measurements["id"] = measurements["sequence"].map(generate_sequence_id)
master_lookup = master[["id", "sequence_normalized"]].rename(
    columns={"id": "master_id", "sequence_normalized": "sequence"}
)
measurements = measurements.merge(
    master_lookup,
    on="sequence",
    how="left",
    validate="many_to_one",
)
measurements["in_maomao_pivot"] = measurements["master_id"].notna()

matched = measurements["in_maomao_pivot"]
assert measurements.loc[matched, "id"].equals(
    measurements.loc[matched, "master_id"].astype(measurements["id"].dtype)
), "A matched sequence has an identifier inconsistent with the master pivot."

measurements["cross_reference_status"] = "matched_master"
measurements.loc[~measurements["is_canonical"], "cross_reference_status"] = "non_canonical"
measurements.loc[
    measurements["is_canonical"] & measurements["sequence_length"].lt(MIN_SEQUENCE_LENGTH),
    "cross_reference_status",
] = "below_minimum_length"
measurements.loc[
    measurements["is_canonical"] & measurements["sequence_length"].gt(MAX_SEQUENCE_LENGTH),
    "cross_reference_status",
] = "above_maximum_length"
measurements.loc[
    ~measurements["in_maomao_pivot"]
    & measurements["is_canonical"]
    & measurements["within_maomao_length"],
    "cross_reference_status",
] = "eligible_sequence_not_in_master"

long_columns = [
    "id",
    "sequence",
    "measurement_type",
    "relation",
    "value",
    "error",
    "unit",
    "raw_label",
    "source",
    "source_dataset",
    "sequence_length",
    "is_canonical",
    "within_maomao_length",
    "in_maomao_pivot",
    "cross_reference_status",
]
measurements_long = measurements[long_columns].sort_values(
    ["measurement_type", "sequence", "unit", "value"],
    kind="stable",
).reset_index(drop=True)

display(measurements_long.head())

,id,sequence,measurement_type,relation,value,error,unit,raw_label,source,source_dataset,sequence_length,is_canonical,within_maomao_length,in_maomao_pivot,cross_reference_status
0,sha256_78fefb7d22cea15fb7b9651175fb09efe6e7574...,FFHHIFRAIVHVAKTIHRLVTG,HC50,=,6.0,NaN,µM,6,Human,Hemolytik2.0_2026,22,True,True,True,matched_master
1,sha256_8c58fd32dc4e67dce0b0ed02793bf1e6322b39c...,FFHHIFRAIVHVGKTIHRLVTG,HC50,=,4.0,NaN,µM,4,Human,Hemolytik2.0_2026,22,True,True,True,matched_master
2,sha256_128eac1031e4c5b4a6ae3564ddeb477ac0095ff...,FFHHIFRAIVHVPKTIHRLVTG,HC50,=,75.0,NaN,µM,75,Human,Hemolytik2.0_2026,22,True,True,True,matched_master
3,sha256_922169e349c4151b0457f1c09e050cc004dc59d...,FFHHIFRGIVHVAKTIHRLVTG,HC50,=,4.0,NaN,µM,4,Human,Hemolytik2.0_2026,22,True,True,True,matched_master
4,sha256_4e1914c54d14afece198fd5d990588c89d4e000...,FFHHIFRGIVHVGKTIHRLVTG,HC50,=,11.0,NaN,µM,11,Human,Hemolytik2.0_2026,22,True,True,True,matched_master


- Cross-reference audit

A sequence can have more than one measurement for the same property, so sequence-level uniqueness is not required in the long table.

In [7]:
def summarize_property(group):
    repeated_groups = (
        group.groupby(["sequence", "unit"], dropna=False)
        .size()
        .gt(1)
        .sum()
    )
    return pd.Series(
        {
            "records": len(group),
            "unique_sequences": group["sequence"].nunique(),
            "matched_records": int(group["in_maomao_pivot"].sum()),
            "matched_unique_sequences": group.loc[
                group["in_maomao_pivot"], "sequence"
            ].nunique(),
            "unmatched_records": int((~group["in_maomao_pivot"]).sum()),
            "unmatched_unique_sequences": group.loc[
                ~group["in_maomao_pivot"], "sequence"
            ].nunique(),
            "repeated_sequence_unit_groups": int(repeated_groups),
            "units": " | ".join(sorted(group["unit"].dropna().unique())),
        }
    )


audit_by_property = (
    measurements.groupby("measurement_type", sort=False)
    .apply(summarize_property, include_groups=False)
    .reset_index()
)
audit_total = pd.DataFrame(
    [
        {
            "measurement_type": "ALL",
            "records": len(measurements),
            "unique_sequences": measurements["sequence"].nunique(),
            "matched_records": int(measurements["in_maomao_pivot"].sum()),
            "matched_unique_sequences": measurements.loc[
                measurements["in_maomao_pivot"], "sequence"
            ].nunique(),
            "unmatched_records": int((~measurements["in_maomao_pivot"]).sum()),
            "unmatched_unique_sequences": measurements.loc[
                ~measurements["in_maomao_pivot"], "sequence"
            ].nunique(),
            "repeated_sequence_unit_groups": int(
                measurements.groupby(["measurement_type", "sequence", "unit"])
                .size()
                .gt(1)
                .sum()
            ),
            "units": " | ".join(sorted(measurements["unit"].dropna().unique())),
        }
    ]
)
audit_cross_reference = pd.concat([audit_by_property, audit_total], ignore_index=True)
unmatched_measurements = measurements_long.loc[
    ~measurements_long["in_maomao_pivot"]
].reset_index(drop=True)

display(audit_cross_reference)
display(unmatched_measurements)

,measurement_type,records,unique_sequences,matched_records,matched_unique_sequences,unmatched_records,unmatched_unique_sequences,repeated_sequence_unit_groups,units
0,HC50,44,43,44,43,0,0,1,µM | µg/mL
1,LC50,59,59,59,59,0,0,0,µM
2,LD50,21,21,15,15,6,6,0,µM
3,MHC,45,45,45,45,0,0,0,µM | µg/mL
4,ALL,169,166,163,160,6,6,1,µM | µg/mL


,id,sequence,measurement_type,relation,value,error,unit,raw_label,source,source_dataset,sequence_length,is_canonical,within_maomao_length,in_maomao_pivot,cross_reference_status
0,sha256_1679ce8528f16f68b882713f3910868045d1661...,MFTLKKSLLFLFFLGTISLSFCEEERGADEDDEVEMTEEEKRSILD...,LD50,=,480.0,NaN,µM,480,Human,Hemolytik2.0_2026,75,True,False,False,above_maximum_length
1,sha256_f4d813401f554d6bb648eb5b29857f5f924c2fe...,MFTMKKSLLFLFFFLGTISLSLCEQERGADEDDGGEEVKRGIFSLF...,LD50,>,600.0,NaN,µM,> 600,Human,Hemolytik2.0_2026,77,True,False,False,above_maximum_length
2,sha256_8af9fa9d2be22a2b7236f57a93c2d9f99604ace...,MFTMKKSLLFLFFLGTISLSFCEEERGADEDDEVEMTEEEKRGVLD...,LD50,>,480.0,NaN,µM,> 480,Human,Hemolytik2.0_2026,75,True,False,False,above_maximum_length
3,sha256_8bd0d3460259dd0f03dba8ab0051affe167d0d3...,MFTMKKSLLLLFFLGIVSLSLCEQERGADEDEGEDVEEVKRSLWEN...,LD50,=,220.0,NaN,µM,220,Human,Hemolytik2.0_2026,72,True,False,False,above_maximum_length
4,sha256_85685f15115381362e134dc641f7ab69fc886f1...,MFTMKKSMLLIVLLGIISLSLCEQERNADEDEQSEMKRISFKKGKG...,LD50,>,480.0,NaN,µM,> 480,Human,Hemolytik2.0_2026,84,True,False,False,above_maximum_length
5,sha256_3aca44e494e3e2e3979fc14164a55c1691f6887...,MFTSKKSMLLLFFLGMISMSLCQDERGADEDDGGEMTEEEKRGAFG...,LD50,=,520.0,NaN,µM,520,Human,Hemolytik2.0_2026,72,True,False,False,above_maximum_length


- Build the sequence-level characterization table

Only measurements matched to the master pivot are summarized here. The resulting table contains the original master columns plus `hc50`, `lc50`, `ld50`, and `mhc`. Each populated cell retains the reported expression and unit, for example `1.4 ± 0.2 µM`, `> 200 µM`, or `7 µM | 125 µM`. The string `999` means that MAOMAO has no numerical measurement for that property. Use the long table for numerical or statistical analysis.

In [8]:
PROPERTY_COLUMNS = ["hc50", "lc50", "ld50", "mhc"]
MISSING_PROPERTY_CODE = "999"

matched_measurements = measurements.loc[measurements["in_maomao_pivot"]].copy()
matched_measurements["property_value"] = (
    matched_measurements["raw_label"].str.strip()
    + " "
    + matched_measurements["unit"].str.strip()
)


def join_unique_measurements(values):
    return " | ".join(dict.fromkeys(str(value) for value in values))


property_pivot = (
    matched_measurements.pivot_table(
        index="id",
        columns="measurement_type",
        values="property_value",
        aggfunc=join_unique_measurements,
        sort=True,
    )
    .rename(columns=str.lower)
    .reset_index()
)

for column in PROPERTY_COLUMNS:
    if column not in property_pivot.columns:
        property_pivot[column] = pd.NA
property_pivot = property_pivot[["id", *PROPERTY_COLUMNS]]
property_pivot["id"] = property_pivot["id"].astype(master["id"].dtype)

sequence_characterization = master[master_original_columns].merge(
    property_pivot,
    on="id",
    how="left",
    validate="one_to_one",
)
sequence_characterization[PROPERTY_COLUMNS] = (
    sequence_characterization[PROPERTY_COLUMNS]
    .fillna(MISSING_PROPERTY_CODE)
    .astype("string")
)
has_measurement = sequence_characterization[PROPERTY_COLUMNS].ne(
    MISSING_PROPERTY_CODE
).any(axis=1)

assert len(sequence_characterization) == len(master)
assert sequence_characterization["id"].equals(master["id"])
assert sequence_characterization["sequence"].equals(master["sequence"])
assert sequence_characterization["id"].is_unique
assert sequence_characterization.columns.tolist() == [
    *master_original_columns,
    *PROPERTY_COLUMNS,
]
assert not sequence_characterization[PROPERTY_COLUMNS].isna().any().any()
assert int(has_measurement.sum()) == matched_measurements["id"].nunique()

print(f"Characterization rows: {len(sequence_characterization):,}")
print(f"Characterization columns: {len(sequence_characterization.columns):,}")
print(
    "Sequences with at least one matched measurement:",
    f"{has_measurement.sum():,}",
)
display(
    sequence_characterization.loc[has_measurement]
    .reset_index(drop=True)
    .head()
)

Characterization rows: 71,857
Characterization columns: 14
Sequences with at least one matched measurement: 160


,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,anti_mammalian_cells,neurotoxic,embryotoxic,ichthyotoxic,hc50,lc50,ld50,mhc
0,sha256_ac92af50ce42fc29a5f93795b96e777e9382fc4...,AAAKAALNAVLVGANA,1,1,1,999,999,999,999,999,999,40 µM,999,999
1,sha256_3b225219a01580b978df1491558586cf57374e0...,ALWDTLLKKVLKAAAKAALDAVLVGANA,1,2,1,999,1,999,999,999,999,5 ± 1 µM,999,999
2,sha256_74d12220e87004fe19eca7bd1905060d242b590...,ALWDTLLKKVLKAAAKAALNAVLVGANA,2,2,1,999,0,999,999,999,999,2.3 ± 0.3 µM,999,999
3,sha256_b13e64545ba214ca025ec8ce956e4598576a848...,ALWKTLLKKVLKAAAK,1,1,1,999,999,999,999,999,999,57 ± 3 µM,999,999
4,sha256_a1fbbb245aced4430ab86ee8acd6527a2a2462e...,ALWKTLLKKVLKAAAKAALKAVLVGANA,1,2,1,999,1,999,999,999,999,0.5 ± 0.1 µM,999,999


- Save outputs and metadata

In [9]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

output_paths = {
    "long_measurements": OUTPUT_ROOT / "maomao_toxicity_measurements.csv",
    "sequence_characterization": (
        OUTPUT_ROOT / "maomao_sequence_pivot_with_toxicity_properties.csv"
    ),
    "cross_reference_audit": (
        OUTPUT_ROOT / "audit_toxicity_property_cross_reference.csv"
    ),
    "unmatched_sequences_audit": (
        OUTPUT_ROOT / "audit_toxicity_property_unmatched_sequences.csv"
    ),
    "metadata": OUTPUT_ROOT / "metadata_toxicity_properties.json",
}

measurements_long.to_csv(output_paths["long_measurements"], index=False)
sequence_characterization.to_csv(output_paths["sequence_characterization"], index=False)
audit_cross_reference.to_csv(output_paths["cross_reference_audit"], index=False)
unmatched_measurements.to_csv(output_paths["unmatched_sequences_audit"], index=False)

metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset": SOURCE_DATASET,
    "identifier_algorithm": "SHA-256",
    "identifier_input": "UTF-8 encoding of the normalized uppercase peptide sequence with whitespace removed",
    "sequence_policy": {
        "canonical_residues": "".join(sorted(CANONICAL_RESIDUES)),
        "minimum_length": MIN_SEQUENCE_LENGTH,
        "maximum_length": MAX_SEQUENCE_LENGTH,
    },
    "unit_policy": "No unit conversion is performed. Each compact property value retains its reported unit.",
    "compact_property_columns": PROPERTY_COLUMNS,
    "compact_missing_value_code": MISSING_PROPERTY_CODE,
    "compact_value_format": "reported expression followed by unit; repeated measurements separated by ' | '",
    "master_rows": int(len(master)),
    "measurement_records": int(len(measurements_long)),
    "measurement_unique_sequences": int(measurements_long["sequence"].nunique()),
    "matched_records": int(measurements_long["in_maomao_pivot"].sum()),
    "matched_unique_sequences": int(
        measurements_long.loc[measurements_long["in_maomao_pivot"], "sequence"].nunique()
    ),
    "unmatched_records": int((~measurements_long["in_maomao_pivot"]).sum()),
    "unmatched_unique_sequences": int(
        measurements_long.loc[~measurements_long["in_maomao_pivot"], "sequence"].nunique()
    ),
    "outputs": {name: path.name for name, path in output_paths.items()},
}

with output_paths["metadata"].open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)
    file.write("\n")

for name, path in output_paths.items():
    print(f"{name}: {path}")

long_measurements: /home/nicole/Descargas/maomao/processed_data/processed_data/maomao_toxicity_measurements.csv
sequence_characterization: /home/nicole/Descargas/maomao/processed_data/processed_data/maomao_sequence_pivot_with_toxicity_properties.csv
cross_reference_audit: /home/nicole/Descargas/maomao/processed_data/processed_data/audit_toxicity_property_cross_reference.csv
unmatched_sequences_audit: /home/nicole/Descargas/maomao/processed_data/processed_data/audit_toxicity_property_unmatched_sequences.csv
metadata: /home/nicole/Descargas/maomao/processed_data/processed_data/metadata_toxicity_properties.json
